# 02 - Collision Type (C) Classification: 5-Prompt + Crop & Zoom Architecture
Specialized notebook for Stage 2 (Collision Type Classification) using the 5-prompt ensemble + entropy-gated pairwise adjudication (Singh et al., arXiv:2606.12047) with 55% Crop & Zoom and 3.5s pre-impact trajectory window.

In [ ]:
# --- 1. Setup & Load Core Pipeline Baseline ---
import sys, os, urllib.request, pathlib, time, re
from collections import Counter
import numpy as np, pandas as pd, cv2
from PIL import Image as PILImage

# Check candidate paths for shared_pipeline.py
possible_paths = [
    '../core/shared_pipeline.py',
    './core/shared_pipeline.py',
    './shared_pipeline.py',
    'zero-shot-cctv-accident/core/shared_pipeline.py'
]

pipeline_path = None
for p in possible_paths:
    if os.path.exists(p):
        pipeline_path = p
        break

if pipeline_path is None:
    print('[STATUS] shared_pipeline.py not found locally. Downloading from branch feature/type-C...')
    urls = [
        'https://raw.githubusercontent.com/HoangDinhBui/zero-shot-cctv-accident/feature/type-C/core/shared_pipeline.py',
        'https://raw.githubusercontent.com/HoangDinhBui/zero-shot-cctv-accident/main/core/shared_pipeline.py'
    ]
    for url in urls:
        try:
            urllib.request.urlretrieve(url, 'shared_pipeline.py')
            pipeline_path = 'shared_pipeline.py'
            print(f'[SUCCESS] Downloaded shared_pipeline.py from {url}')
            break
        except Exception as e:
            print(f'[WARNING] Failed downloading from {url}: {e}')

if pipeline_path:
    p_dir = os.path.dirname(pipeline_path)
    if p_dir and p_dir not in sys.path:
        sys.path.append(p_dir)
    print(f'[STATUS] Running core pipeline from: {pipeline_path}')
    %run $pipeline_path
else:
    print('[ERROR] shared_pipeline.py not found! Please check repository or network.')


In [ ]:
# --- 2. Temporal Anchors & Scope Helpers ---
def predict_accident_time(video_path, smooth_window=5, z_threshold=1.5):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if fps <= 0 or n_frames == 0:
        return 0.0
    diff_series = compute_frame_diff_series(video_path)
    if len(diff_series) == 0:
        return n_frames / fps / 2.0
    anomaly = score_temporal_anomaly(diff_series, smooth_window)
    candidates = np.where(anomaly > z_threshold)[0]
    peak_frame = int(np.argmax(anomaly)) if len(candidates) == 0 else int(candidates[np.argmax(anomaly[candidates])])
    return round(peak_frame / fps, 4)

def predict_accident_time_ensemble(video_path, smooth_window=5, z_threshold=1.5):
    return predict_accident_time(video_path, smooth_window, z_threshold)

print('[STATUS] Temporal anchor helpers defined')


## 3. Five-Prompt + Entropy-Gated Pairwise Adjudicator (Singh et al., arXiv:2606.12047)

**Kiến trúc 5-Prompt với 3 cải tiến lớn cho Stage 2 (Type C):**
1. **5-Prompt complementary views:** p1 Baseline, p2 Temporal motion, p3 Contact geometry, p4 Contrastive elimination, p5 Tiebreaker, Pairwise adjudicator.
2. **Crop & Zoom 55% quanh tâm va chạm (center_x, center_y):** Tăng mật độ điểm ảnh của 2 xe lên gấp 4 lần, loại bỏ 70% nhiễu nền từ các xe khác.
3. **Cửa sổ quỹ đạo tiếp cận [t_final - 3.5s đến t_final + 0.5s]:** Cho phép model thấy rõ hướng di chuyển ban đầu (cùng chiều, ngược chiều, cắt ngang) thay vì chỉ thấy đống đổ nát sau va chạm.

In [ ]:
# --- 4. 5-Prompt Constants, Templates, Metadata Builder & Parser ---
# Refined with: Vehicle Interaction Disambiguation & Pre/Post Impact Separation (DRAMA WACV 2023 / Singh et al.)

METADATA_BY_PATH_5P = {}
if 'test_df' in globals() and not test_df.empty:
    _meta_cols_5p = [c for c in ['scene_layout', 'lighting', 'weather'] if c in test_df.columns]
    if _meta_cols_5p and 'path' in test_df.columns:
        for _, _row in test_df.iterrows():
            if pd.notna(_row.get('path')):
                METADATA_BY_PATH_5P[_row['path']] = {
                    c: _row[c] for c in _meta_cols_5p if pd.notna(_row.get(c))
                }
print(f'[STATUS] METADATA_BY_PATH_5P: {len(METADATA_BY_PATH_5P)} entries')

def _build_metadata_context_5p(sub_path):
    meta = METADATA_BY_PATH_5P.get(sub_path, {})
    if not meta:
        return ''
    parts = []
    if 'scene_layout' in meta:
        parts.append(f"Scene: {meta['scene_layout'].replace('_', ' ')}")
    if 'weather' in meta:
        parts.append(f"Weather: {meta['weather']}")
    if 'lighting' in meta:
        parts.append(f"Time of day: {meta['lighting']}")
    return 'Video context: ' + '. '.join(parts) + '.\n' if parts else ''

CATEGORIES_5P = (
    "A. rear-end -- a following vehicle strikes the one ahead, both traveling the SAME direction\n"
    "B. t-bone -- the front of one vehicle strikes the side of another at approximately 90 degrees\n"
    "C. single -- exactly one vehicle is involved (hits wall, pole, barrier, or loses control)\n"
    "D. head-on -- two vehicles collide front-to-front, traveling in OPPOSITE directions\n"
    "E. sideswipe -- two roughly parallel vehicles make glancing side-to-side contact"
)

# Enhanced output constraint: forces identification of the exact 2 colliding vehicles
OUTPUT_CONSTRAINT_5P = (
    "CRITICAL RULES:\n"
    "- Focus ONLY on the two vehicles that directly COLLIDE/STOP. Ignore passing bystander vehicles.\n"
    "- Check frames 1-3 to see original approach directions (same lane, opposite lane, or crossroad).\n"
    "- Do NOT judge solely by post-crash rotation.\n\n"
    "REQUIRED format -- follow this EXACTLY:\n"
    "1. Vehicles: Describe ONLY the 2 vehicles involved in the collision.\n"
    "2. Contact: Describe the exact contact point and pre-crash directions.\n"
    "3. Answer: <write one label: rear-end, t-bone, single, head-on, or sideswipe>"
)

TYPE_DEFS_5P = {
    'rear-end': 'a following vehicle strikes the one ahead, both traveling the same direction',
    't-bone': 'the front of one vehicle strikes the side of another at approximately 90 degrees',
    'single': 'exactly one vehicle is involved (hits wall, pole, barrier, or loses control)',
    'head-on': 'two vehicles collide front-to-front, traveling in opposite directions',
    'sideswipe': 'two roughly parallel vehicles make glancing side-to-side contact',
}

def _prompt_p1_baseline(meta_ctx):
    return f"You are given frames from a traffic accident video.\n{meta_ctx}Analyze the frames and classify the accident type.\n\nCategories:\n{CATEGORIES_5P}\n\n{OUTPUT_CONSTRAINT_5P}"

def _prompt_p2_temporal_motion(meta_ctx):
    return f"You are given chronological frames from a traffic accident.\n{meta_ctx}Track the motion of the colliding vehicles from the first frame to impact.\n\nCategories:\n{CATEGORIES_5P}\n\nRules:\n- Track original travel lanes in early frames to determine if vehicles were opposite (head-on), same (rear-end), or cross (t-bone).\n\n{OUTPUT_CONSTRAINT_5P}"

def _prompt_p3_contact_geometry(meta_ctx):
    return f"You are given zoomed frames of a traffic accident.\n{meta_ctx}Focus on the contact geometry:\n- Which parts meet (front-to-front, front-to-back, front-to-side, side-to-side)?\n\nCategories:\n{CATEGORIES_5P}\n\n{OUTPUT_CONSTRAINT_5P}"

def _prompt_p4_contrastive_elimination(meta_ctx):
    return f"You are given frames from a traffic accident video.\n{meta_ctx}Classify by elimination:\n- Opposite lanes colliding front-to-front -> head-on\n- Same lane following vehicle hitting leading vehicle -> rear-end\n- Perpendicular intersection collision -> t-bone\n- Side-to-side glancing contact -> sideswipe\n- Single car crashing alone -> single\n\n{OUTPUT_CONSTRAINT_5P}"

def _prompt_p5_tiebreaker(meta_ctx):
    return f"You are given a traffic accident video.\n{meta_ctx}Carefully adjudicate between colliding vehicles:\n1. Count vehicles directly involved.\n2. Determine pre-impact directions.\n3. Identify first contact point.\n\nCategories:\n{CATEGORIES_5P}\n\n{OUTPUT_CONSTRAINT_5P}"

def _prompt_pairwise_adjudicator(meta_ctx, c1, c2):
    return f"You are given a traffic accident video.\n{meta_ctx}Final binary choice: choose ONLY between {c1} and {c2}.\n\n- {c1}: {TYPE_DEFS_5P.get(c1, c1)}\n- {c2}: {TYPE_DEFS_5P.get(c2, c2)}\n\n{OUTPUT_CONSTRAINT_5P}"

LETTER_TO_TYPE_5P = {'a': 'rear-end', 'b': 't-bone', 'c': 'single', 'd': 'head-on', 'e': 'sideswipe'}

def _parse_type_from_structured(text):
    if not text:
        return None
    m = re.search(r'(?:3\.?\s*)?[Aa]nswer\s*:\s*(.+)', text)
    if m:
        ans = m.group(1).strip()
        if ans and ans[0].lower() in LETTER_TO_TYPE_5P:
            return LETTER_TO_TYPE_5P[ans[0].lower()]
        lbl = _parse_type_label(ans)
        if lbl:
            return lbl
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if lines:
        last = lines[-1]
        if last and last[0].lower() in LETTER_TO_TYPE_5P and len(last) < 30:
            return LETTER_TO_TYPE_5P[last[0].lower()]
        lbl = _parse_type_label(last)
        if lbl:
            return lbl
    return _parse_type_label(text)

def _normalized_entropy_5p(counts):
    total = sum(counts.values())
    if total == 0:
        return 0.0
    K = len([c for c in counts.values() if c > 0])
    if K <= 1:
        return 0.0
    probs = [c / total for c in counts.values() if c > 0]
    H = -sum(p * np.log2(p) for p in probs)
    return float(H / np.log2(K))

print('[STATUS] Enhanced prompts with vehicle disambiguation ready')


In [ ]:
# --- 5. Crop & Zoom + Trajectory Sampling + 5-Prompt Classifier ---
TAU_M_5P = 2
TAU_H_5P = 0.75
MAX_NEW_TOKENS_5P = 128

def refine_center_via_flow(video_path, t_final, center_x=0.5, center_y=0.5):
    '''If center coordinate is default/uncertain, refine using Optical Flow energy centroid around t_final.'''
    if center_x != 0.5 or center_y != 0.5:
        return center_x, center_y
    try:
        flow_map = compute_flow_magnitude_map(video_path, resize_w=320, resize_h=180, n_frames_context=20, flow_percentile=90.0)
        total_e = flow_map.sum()
        if total_e > 1e-4:
            y_indices, x_indices = np.indices(flow_map.shape)
            fx = float((x_indices * flow_map).sum() / (total_e * flow_map.shape[1]))
            fy = float((y_indices * flow_map).sum() / (total_e * flow_map.shape[0]))
            return fx, fy
    except Exception:
        pass
    return center_x, center_y

def crop_around_center(pil_img, center_x=0.5, center_y=0.5, crop_frac=0.55):
    if pil_img is None:
        return None
    W, H = pil_img.size
    cx = 0.5 if (center_x is None or np.isnan(center_x)) else float(np.clip(center_x, 0.0, 1.0))
    cy = 0.5 if (center_y is None or np.isnan(center_y)) else float(np.clip(center_y, 0.0, 1.0))

    cw, ch = int(W * crop_frac), int(H * crop_frac)
    px, py = int(cx * W), int(cy * H)

    x1 = max(0, min(W - cw, px - cw // 2))
    y1 = max(0, min(H - ch, py - ch // 2))
    x2 = min(W, x1 + cw)
    y2 = min(H, y1 + ch)
    return pil_img.crop((x1, y1, x2, y2))

def sample_frames_for_type_classification(video_path, t_final, center_x=0.5, center_y=0.5, n_frames=6, crop_frac=0.55):
    cx, cy = refine_center_via_flow(video_path, t_final, center_x, center_y)
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or n_total <= 0:
        cap.release()
        return []
    duration = n_total / fps
    t_start, t_end = max(0.0, t_final - 3.5), min(duration, t_final + 0.5)
    timestamps = np.linspace(t_start, t_end, n_frames)
    out_frames = []

    for t in timestamps:
        frame_idx = min(int(round(t * fps)), n_total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = PILImage.fromarray(rgb)
            cropped = crop_around_center(pil_img, cx, cy, crop_frac=crop_frac)
            cropped.thumbnail((VLM_CFG['image_max_side'], VLM_CFG['image_max_side']), PILImage.Resampling.LANCZOS)
            out_frames.append((round(t, 2), cropped))
    cap.release()
    return out_frames

def classify_type_5prompt_v2(video_path, t_final, type_from_stage1, center_x=0.5, center_y=0.5, scene_layout=None, sub_path=None):
    if not VLM_AVAILABLE:
        return type_from_stage1

    frames = sample_frames_for_type_classification(video_path, t_final, center_x=center_x, center_y=center_y, n_frames=6, crop_frac=0.55)
    if not frames:
        frames = sample_frames_stamped(video_path, 2.0, t_final - 1.5, t_final + 1.0, limit=6, max_side=VLM_CFG['image_max_side'], burn=False)
    if not frames:
        return type_from_stage1

    _sub = sub_path or ('videos/' + pathlib.Path(video_path).name)
    meta_ctx = _build_metadata_context_5p(_sub)

    votes = [type_from_stage1]
    base_prompt_fns = [_prompt_p1_baseline, _prompt_p2_temporal_motion, _prompt_p3_contact_geometry, _prompt_p4_contrastive_elimination]

    for fn in base_prompt_fns:
        prompt = fn(meta_ctx)
        raw = vlm_generate_oom_safe(prompt, frames, MAX_NEW_TOKENS_5P, min_frames=2)
        label = _parse_type_from_structured(raw)
        if label is not None:
            votes.append(label)

    counts = Counter(votes)
    top_label, top_n = counts.most_common(1)[0]
    second_n = counts.most_common(2)[1][1] if len(counts) > 1 else 0
    margin = top_n - second_n
    H_norm = _normalized_entropy_5p(counts)

    print(f'  [5P+Crop] Base votes: {dict(counts)} | margin={margin} H={H_norm:.3f}')
    if margin >= TAU_M_5P or H_norm <= TAU_H_5P:
        print(f'  [5P+Crop] -> EXIT base: {top_label}')
        return top_label

    esc_frames = frames[::max(1, len(frames) // 6)][:6]
    raw = vlm_generate_oom_safe(_prompt_p5_tiebreaker(meta_ctx), esc_frames, MAX_NEW_TOKENS_5P, min_frames=2)
    label = _parse_type_from_structured(raw)
    if label is not None:
        votes.append(label)

    counts = Counter(votes)
    top_label, top_n = counts.most_common(1)[0]
    second_n = counts.most_common(2)[1][1] if len(counts) > 1 else 0
    margin = top_n - second_n
    H_norm = _normalized_entropy_5p(counts)

    print(f'  [5P+Crop] After tiebreaker: {dict(counts)} | margin={margin} H={H_norm:.3f}')
    if margin >= TAU_M_5P or H_norm <= TAU_H_5P:
        print(f'  [5P+Crop] -> EXIT tiebreaker: {top_label}')
        return top_label

    top2 = counts.most_common(2)
    if len(top2) < 2:
        return top_label

    c1, c2 = top2[0][0], top2[1][0]
    raw = vlm_generate_oom_safe(_prompt_pairwise_adjudicator(meta_ctx, c1, c2), esc_frames, MAX_NEW_TOKENS_5P, min_frames=2)
    label = _parse_type_from_structured(raw)
    result = label if label in (c1, c2) else top_label
    print(f'  [5P+Crop] -> EXIT pairwise ({c1} vs {c2}): adj={label} -> {result}')
    return result

print('[STATUS] Enhanced classifier ready')


In [ ]:
# --- 6. Pipeline Definitions (Baseline vs Experiment) ---
def run_inference_vlm(video_path: pathlib.Path, sub_path: str = None) -> dict:
    '''Baseline pipeline (core): uses classify_type_cascade.'''
    if not VLM_AVAILABLE:
        raise RuntimeError('run_inference_vlm called with no usable VLM')
    video_path = pathlib.Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    scene = SCENE_BY_PATH.get(sub_path or ('videos/' + video_path.name))

    pred = stage1_full_scan(video_path, duration, scene)
    global PARSE_FAIL_STREAK
    if pred.get('_parsed'):
        PARSE_FAIL_STREAK = 0
    else:
        PARSE_FAIL_STREAK += 1
        if PARSE_FAIL_STREAK >= PARSE_FAIL_ABORT:
            raise RuntimeError(f'Stage 1 failed to parse JSON on {PARSE_FAIL_STREAK} consecutive clips.')

    t_classical = predict_accident_time_ensemble(video_path)
    t_vlm = pred['accident_time']
    correction = float(np.clip(t_vlm - t_classical, -TEMPORAL_ANCHOR_DELTA_MAX, TEMPORAL_ANCHOR_DELTA_MAX))
    pred['accident_time'] = float(np.clip(t_classical + correction, 0.0, duration))

    t_final = stage2_time_refine(video_path, pred['accident_time'], duration)
    pred['accident_time'] = t_final

    pt = stage3_grounding(video_path, t_final)
    if pt is not None:
        pred['center_x'], pred['center_y'] = pt

    pred['type'] = classify_type_cascade(video_path, t_final, pred['type'])
    pred['type'] = apply_scene_type_postfix(pred['type'], scene)

    return {'path': str(video_path), 'accident_time': pred['accident_time'],
            'center_x': pred['center_x'], 'center_y': pred['center_y'],
            'type': pred['type'], 'scene_layout': scene}


def run_inference_vlm_C_experiment(video_path: pathlib.Path, sub_path: str = None) -> dict:
    '''Experiment pipeline: uses classify_type_5prompt_v2 with Crop & Zoom around (center_x, center_y).'''
    if not VLM_AVAILABLE:
        raise RuntimeError('run_inference_vlm_C_experiment called with no usable VLM')
    video_path = pathlib.Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = (n / fps) if fps > 0 else 20.0

    scene = SCENE_BY_PATH.get(sub_path or ('videos/' + video_path.name))

    pred = stage1_full_scan(video_path, duration, scene)
    global PARSE_FAIL_STREAK
    if pred.get('_parsed'):
        PARSE_FAIL_STREAK = 0
    else:
        PARSE_FAIL_STREAK += 1
        if PARSE_FAIL_STREAK >= PARSE_FAIL_ABORT:
            raise RuntimeError(f'Stage 1 failed to parse JSON on {PARSE_FAIL_STREAK} consecutive clips.')

    t_classical = predict_accident_time_ensemble(video_path)
    t_vlm = pred['accident_time']
    correction = float(np.clip(t_vlm - t_classical, -TEMPORAL_ANCHOR_DELTA_MAX, TEMPORAL_ANCHOR_DELTA_MAX))
    pred['accident_time'] = float(np.clip(t_classical + correction, 0.0, duration))

    t_final = stage2_time_refine(video_path, pred['accident_time'], duration)
    pred['accident_time'] = t_final

    pt = stage3_grounding(video_path, t_final)
    if pt is not None:
        pred['center_x'], pred['center_y'] = pt

    pred['type'] = classify_type_5prompt_v2(
        video_path, t_final, pred['type'],
        center_x=pred['center_x'], center_y=pred['center_y'],
        scene_layout=scene,
        sub_path=sub_path or ('videos/' + video_path.name),
    )
    pred['type'] = apply_scene_type_postfix(pred['type'], scene)

    return {'path': str(video_path), 'accident_time': pred['accident_time'],
            'center_x': pred['center_x'], 'center_y': pred['center_y'],
            'type': pred['type'], 'scene_layout': scene}

print('[STATUS] Pipelines defined (Baseline & 5-Prompt Experiment)')


In [ ]:
# --- 7. Evaluation & A/B Comparison on Diverse Calibration Set ---
results_baseline = []
results_5prompt = []

print(f'[EVAL] Running evaluation on {len(diverse_videos)} diverse calibration videos...')

for i, vp in enumerate(diverse_videos):
    sub_path = 'videos/' + vp.name
    print(f'\n=== [{i+1}/{len(diverse_videos)}] {vp.name} ===')

    # Baseline (core pipeline)
    t0 = time.time()
    pred_base = run_inference_vlm(vp, sub_path)
    dt_base = time.time() - t0
    results_baseline.append(pred_base)

    # 5-prompt experiment
    t0 = time.time()
    pred_5p = run_inference_vlm_C_experiment(vp, sub_path)
    dt_5p = time.time() - t0
    results_5prompt.append(pred_5p)

    changed = ' <<< CHANGED' if pred_base['type'] != pred_5p['type'] else ''
    print(f'  Baseline: type={pred_base["type"]:12s} ({dt_base:.1f}s)')
    print(f'  5-prompt: type={pred_5p["type"]:12s} ({dt_5p:.1f}s){changed}')

# Summary Statistics
gt_types = list(diverse_labels_df['type']) if 'type' in diverse_labels_df.columns else []
base_types = [r['type'] for r in results_baseline]
exp_types = [r['type'] for r in results_5prompt]

n_changed = sum(1 for b, p in zip(base_types, exp_types) if b != p)

print('\n' + '='*50)
print('                  SUMMARY RESULTS                  ')
print('='*50)
print(f'Total calibration videos: {len(diverse_videos)}')
print(f'Type predictions changed: {n_changed}/{len(diverse_videos)} ({n_changed/len(diverse_videos):.0%})')
print(f'Baseline distribution:   {dict(Counter(base_types))}')
print(f'5-Prompt distribution:   {dict(Counter(exp_types))}')

if gt_types:
    base_acc = np.mean([b == g for b, g in zip(base_types, gt_types)])
    exp_acc = np.mean([e == g for e, g in zip(exp_types, gt_types)])
    print(f'Ground Truth distribution: {dict(Counter(gt_types))}')
    print(f'Baseline Type Accuracy:    {base_acc:.4f}')
    print(f'5-Prompt Type Accuracy:    {exp_acc:.4f} ({exp_acc - base_acc:+.4f})')
print('='*50)
